# Phone Unlock State Machine Audit

This notebook inspects `data/android/unlock-events.jsonl` and checks whether the expected phone interval state machine is sane before importing it into Behaviour Hub.

The two candidate strategies are:

- `keyguard`: `KEYGUARD_HIDDEN` starts `Screen / Screen on`, `KEYGUARD_SHOWN` ends it.
- `screen`: `SCREEN_INTERACTIVE` starts `Screen / Screen on`, `SCREEN_NON_INTERACTIVE` ends it.

The importer currently repairs repeated starts by replacing the earlier open start with the later start.

In [1]:
from __future__ import annotations

import json
from collections import Counter, defaultdict
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path

import polars as pl

from timeline.domain.enums import EventKind
from timeline.ingest.android import import_android_unlock_jsonl, parse_android_timestamp

In [2]:
PATH = Path("../data/android/unlock-events.jsonl")

rows = []
with PATH.open(encoding="utf-8") as file:
    for line_number, line in enumerate(file, start=1):
        if not line.strip():
            continue
        row = json.loads(line)
        row["line_number"] = line_number
        row["parsed_timestamp"] = parse_android_timestamp(row["timestamp"])
        rows.append(row)

df = pl.DataFrame(rows)
df.select([
    pl.len().alias("rows"),
    pl.col("parsed_timestamp").min().alias("first_timestamp"),
    pl.col("parsed_timestamp").max().alias("last_timestamp"),
])

rows,first_timestamp,last_timestamp
u32,datetime[μs],datetime[μs]
2802,2026-07-26 18:15:27.403,2026-08-02 17:41:13.171


In [3]:
(
    df
    .group_by(["event_type", "event"])
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
)

event_type,event,count
i64,str,u32
16,"""SCREEN_NON_INTERACTIVE""",760
15,"""SCREEN_INTERACTIVE""",760
17,"""KEYGUARD_SHOWN""",637
18,"""KEYGUARD_HIDDEN""",635
27,"""DEVICE_STARTUP""",5
26,"""DEVICE_SHUTDOWN""",5


## State Machine Check

For each strategy, the desired interval stream is simply:

`START, END, START, END, ...`

Bad transitions are:

- `duplicate_start`: another start appears while an interval is already open.
- `orphan_end`: an end appears while no interval is open.
- `open_at_eof`: the file ends with an interval still open.

The importer repairs `duplicate_start` by replacing the earlier open start with the later one.

In [4]:
STRATEGIES = {
    "keyguard": {
        "KEYGUARD_HIDDEN": EventKind.INTERVAL_START,
        "KEYGUARD_SHOWN": EventKind.INTERVAL_END,
    },
    "screen": {
        "SCREEN_INTERACTIVE": EventKind.INTERVAL_START,
        "SCREEN_NON_INTERACTIVE": EventKind.INTERVAL_END,
    },
}


@dataclass(frozen=True)
class StateMachineScan:
    strategy: str
    intervals: list[dict]
    anomalies: list[dict]
    ignored_events: int


def scan_state_machine(strategy: str, *, repair_duplicate_starts: bool) -> StateMachineScan:
    event_map = STRATEGIES[strategy]
    intervals = []
    anomalies = []
    ignored_events = 0
    open_start = None

    for row in rows:
        event_kind = event_map.get(row["event"])
        if event_kind is None:
            ignored_events += 1
            continue

        if event_kind == EventKind.INTERVAL_START:
            if open_start is not None:
                anomalies.append({
                    "kind": "duplicate_start",
                    "strategy": strategy,
                    "line_number": row["line_number"],
                    "timestamp": row["parsed_timestamp"],
                    "event": row["event"],
                    "open_line_number": open_start["line_number"],
                    "open_timestamp": open_start["parsed_timestamp"],
                    "open_event": open_start["event"],
                })
                if repair_duplicate_starts:
                    open_start = row
                continue

            open_start = row
            continue

        if open_start is None:
            anomalies.append({
                "kind": "orphan_end",
                "strategy": strategy,
                "line_number": row["line_number"],
                "timestamp": row["parsed_timestamp"],
                "event": row["event"],
            })
            continue

        intervals.append({
            "strategy": strategy,
            "start_line_number": open_start["line_number"],
            "end_line_number": row["line_number"],
            "start_timestamp": open_start["parsed_timestamp"],
            "end_timestamp": row["parsed_timestamp"],
            "duration_seconds": int((row["parsed_timestamp"] - open_start["parsed_timestamp"]).total_seconds()),
            "start_event": open_start["event"],
            "end_event": row["event"],
        })
        open_start = None

    if open_start is not None:
        anomalies.append({
            "kind": "open_at_eof",
            "strategy": strategy,
            "line_number": open_start["line_number"],
            "timestamp": open_start["parsed_timestamp"],
            "event": open_start["event"],
        })

    return StateMachineScan(
        strategy=strategy,
        intervals=intervals,
        anomalies=anomalies,
        ignored_events=ignored_events,
    )

In [5]:
summary_rows = []
scans = {}

for strategy in STRATEGIES:
    raw_scan = scan_state_machine(strategy, repair_duplicate_starts=False)
    repaired_scan = scan_state_machine(strategy, repair_duplicate_starts=True)
    importer_result = import_android_unlock_jsonl(PATH, strategy=strategy)
    scans[(strategy, "raw")] = raw_scan
    scans[(strategy, "repaired")] = repaired_scan

    anomaly_counts = Counter(anomaly["kind"] for anomaly in raw_scan.anomalies)
    repaired_anomaly_counts = Counter(anomaly["kind"] for anomaly in repaired_scan.anomalies)
    summary_rows.append({
        "strategy": strategy,
        "raw_intervals": len(raw_scan.intervals),
        "repaired_intervals": len(repaired_scan.intervals),
        "ignored_events": raw_scan.ignored_events,
        "raw_duplicate_starts": anomaly_counts["duplicate_start"],
        "raw_orphan_ends": anomaly_counts["orphan_end"],
        "raw_open_at_eof": anomaly_counts["open_at_eof"],
        "repaired_duplicate_starts": repaired_anomaly_counts["duplicate_start"],
        "repaired_orphan_ends": repaired_anomaly_counts["orphan_end"],
        "repaired_open_at_eof": repaired_anomaly_counts["open_at_eof"],
        "importer_events": len(importer_result.events),
        "importer_dropped_orphan_starts": importer_result.dropped_orphan_starts,
    })

pl.DataFrame(summary_rows)

strategy,raw_intervals,repaired_intervals,ignored_events,raw_duplicate_starts,raw_orphan_ends,raw_open_at_eof,repaired_duplicate_starts,repaired_orphan_ends,repaired_open_at_eof,importer_events,importer_dropped_orphan_starts
str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
"""keyguard""",634,634,1530,0,3,1,0,3,1,1272,0
"""screen""",757,757,1282,2,3,1,2,3,1,1518,2


## Duration Audit

Large durations are usually where state-machine mistakes show up first.

In [6]:
duration_rows = []

for strategy in STRATEGIES:
    for mode in ["raw", "repaired"]:
        intervals = scans[(strategy, mode)].intervals
        if not intervals:
            continue
        interval_df = pl.DataFrame(intervals)
        duration_rows.append({
            "strategy": strategy,
            "mode": mode,
            "intervals": interval_df.height,
            "total_hours": interval_df["duration_seconds"].sum() / 3600,
            "median_seconds": interval_df["duration_seconds"].median(),
            "p95_seconds": interval_df["duration_seconds"].quantile(0.95),
            "max_seconds": interval_df["duration_seconds"].max(),
        })

pl.DataFrame(duration_rows).sort(["strategy", "mode"])

strategy,mode,intervals,total_hours,median_seconds,p95_seconds,max_seconds
str,str,i64,f64,f64,f64,i64
"""keyguard""","""raw""",634,46.062222,39.0,1478.0,7488
"""keyguard""","""repaired""",634,46.062222,39.0,1478.0,7488
"""screen""","""raw""",757,70.048611,27.0,1478.0,25190
"""screen""","""repaired""",757,64.580833,27.0,1471.0,25190


In [7]:
def longest_intervals(strategy: str, mode: str, n: int = 20) -> pl.DataFrame:
    intervals = scans[(strategy, mode)].intervals
    if not intervals:
        return pl.DataFrame()
    return (
        pl.DataFrame(intervals)
        .sort("duration_seconds", descending=True)
        .head(n)
    )

longest_intervals("keyguard", "raw", 20)

strategy,start_line_number,end_line_number,start_timestamp,end_timestamp,duration_seconds,start_event,end_event
str,i64,i64,datetime[μs],datetime[μs],i64,str,str
"""keyguard""",2469,2472,2026-08-01 21:37:04.774,2026-08-01 23:41:53.376,7488,"""KEYGUARD_HIDDEN""","""KEYGUARD_SHOWN"""
"""keyguard""",2748,2750,2026-08-02 12:48:36.862,2026-08-02 14:13:41.161,5104,"""KEYGUARD_HIDDEN""","""KEYGUARD_SHOWN"""
"""keyguard""",514,516,2026-07-28 13:17:17.644,2026-07-28 14:38:27.538,4869,"""KEYGUARD_HIDDEN""","""KEYGUARD_SHOWN"""
"""keyguard""",2135,2137,2026-07-31 23:51:51.764,2026-08-01 01:10:24.771,4713,"""KEYGUARD_HIDDEN""","""KEYGUARD_SHOWN"""
"""keyguard""",2293,2295,2026-08-01 16:07:41.605,2026-08-01 17:24:55.050,4633,"""KEYGUARD_HIDDEN""","""KEYGUARD_SHOWN"""
…,…,…,…,…,…,…,…
"""keyguard""",306,308,2026-07-27 15:37:00.078,2026-07-27 16:17:08.958,2408,"""KEYGUARD_HIDDEN""","""KEYGUARD_SHOWN"""
"""keyguard""",448,451,2026-07-27 23:27:19.202,2026-07-28 00:05:55.710,2316,"""KEYGUARD_HIDDEN""","""KEYGUARD_SHOWN"""
"""keyguard""",2213,2215,2026-08-01 12:47:56.799,2026-08-01 13:24:08.061,2171,"""KEYGUARD_HIDDEN""","""KEYGUARD_SHOWN"""


In [8]:
def anomalies(strategy: str, mode: str = "raw") -> pl.DataFrame:
    items = scans[(strategy, mode)].anomalies
    return pl.DataFrame(items) if items else pl.DataFrame()

anomalies("keyguard")

kind,strategy,line_number,timestamp,event
str,str,i64,datetime[μs],str
"""orphan_end""","""keyguard""",2,2026-07-26 18:15:27.973,"""KEYGUARD_SHOWN"""
"""orphan_end""","""keyguard""",43,2026-07-26 20:37:10.915,"""KEYGUARD_SHOWN"""
"""orphan_end""","""keyguard""",1245,2026-07-29 23:49:00.795,"""KEYGUARD_SHOWN"""
"""open_at_eof""","""keyguard""",2802,2026-08-02 17:41:13.171,"""KEYGUARD_HIDDEN"""


## Inspect Rows Around Anomalies

Use this to inspect the raw event sequence around duplicate starts or orphan ends.

In [9]:
def rows_around(line_number: int, radius: int = 6) -> pl.DataFrame:
    start = max(1, line_number - radius)
    end = line_number + radius
    return (
        df
        .filter(pl.col("line_number").is_between(start, end))
        .select([
            "line_number",
            "id",
            "parsed_timestamp",
            "event_type",
            "event",
            "package_name",
            "class_name",
        ])
    )

keyguard_anomalies = anomalies("keyguard")
if keyguard_anomalies.is_empty():
    print("No keyguard anomalies found.")
else:
    rows_around(keyguard_anomalies["line_number"][0])

## Importer Parity Check

These checks compare the notebook's repaired state-machine scan with the actual importer.

In [10]:
for strategy in STRATEGIES:
    importer_result = import_android_unlock_jsonl(PATH, strategy=strategy)
    repaired_scan = scans[(strategy, "repaired")]
    repaired_event_count = len(repaired_scan.intervals) * 2
    if any(anomaly["kind"] == "open_at_eof" for anomaly in repaired_scan.anomalies):
        repaired_event_count += 1

    print(strategy)
    print("  importer events:", len(importer_result.events))
    print("  repaired scan events:", repaired_event_count)
    print("  importer dropped orphan starts:", importer_result.dropped_orphan_starts)
    print("  raw duplicate starts:", sum(1 for anomaly in scans[(strategy, "raw")].anomalies if anomaly["kind"] == "duplicate_start"))

keyguard
  importer events: 1272
  repaired scan events: 1269
  importer dropped orphan starts: 0
  raw duplicate starts: 0
screen
  importer events: 1518
  repaired scan events: 1515
  importer dropped orphan starts: 2
  raw duplicate starts: 2
